[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P07/P07-1_Lire_Nettoyer.ipynb)

# Lecture et nettoyage des mesures

Chapitre 7 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Le service qualité vous remet un classeur Excel de mesures, un fichier de pannes de dix mille relevés, un suivi de production de vingt mille lignes. Combien de lignes et de colonnes, de quel type, combien de cases vides, et chaque mesure reste-t-elle physiquement possible ? Vous voudriez le savoir avant de calculer quoi que ce soit, et sans ouvrir le classeur à la main ?

**Objectif terminal.** Ce chapitre montre comment lire ces fichiers, leur donner un type, compter ce qui manque et contrôler la cohérence, avec la bibliothèque pandas.

**Objectif instrumental.**

- Lire un classeur Excel ou un CSV français dans un tableau pandas.
- Choisir des colonnes et des lignes par leur nom, leur adresse ou un masque.
- Typer chaque colonne sans toucher à l'original, compter les valeurs manquantes, contrôler doublons et plages physiques.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 4 et 6 pour les masques et les formes de tableaux, après une filière scientifique.

## Source de données

| Fichier | Lignes | Colonnes | Unités |
|:-|-:|-:|:-|
| `MecaIA-RUL.xlsx` | 240 | 6 | couple en N·m, puissance en kW, masse en kg, temps de réponse en ms, durée de vie résiduelle en jours |
| `MecaIA-Suivi_Production_Machines.xlsx` | 20000 | 14 | ancienneté en années, complexité et efficacité sans dimension, nombre de capteurs entier, utilisation en proportion, température en °C, vibrations en mm/s, consommation en kWh/h, productivité en pièces/h, indice de santé de 0 à 100 |
| `MecaIA-AI4I_maintenance.xlsx` | 10000 | 9 | temps en jours, températures en °C, vitesse en tr/min, couple en N·m, usure en h, puissance en W, vibration en mm/s |
| `MecaIA-AI4I_maintenance_TypePanne.xlsx` | 10000 | 7 | temps en jours, indicateurs 0 ou 1 |

Les classeurs sont lus dans le dossier `donnees` du dépôt `Meca-PY_Ed2026` ; ouvert seul, par exemple dans Colab, le notebook les télécharge une fois depuis le dépôt public.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 7.1 | Lire un fichier de mesures | 10 min |
| 7.2 | Typer une colonne et choisir des lignes | 15 min |
| 7.3 | Cellules manquantes du suivi de production | 15 min |
| 7.4 | Trois cent trente-neuf pannes, trois cent soixante-treize modes (défi) | 25 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### Lecture des jeux

Les jeux se cherchent d'abord dans le dépôt cloné, à deux dossiers au-dessus de ce notebook.

In [ ]:
from pathlib import Path
import urllib.request

# Les jeux du chapitre : nom du classeur et adresse de la copie publique.
JEUX = {
    "MecaIA-RUL.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-RUL.xlsx",
    "MecaIA-Suivi_Production_Machines.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-Suivi_Production_Machines.xlsx",
    "MecaIA-AI4I_maintenance.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-AI4I_maintenance.xlsx",
    "MecaIA-AI4I_maintenance_TypePanne.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-AI4I_maintenance_TypePanne.xlsx",
}

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-PY_Ed2026", "donnees")]
dossier = next((d for d in candidats if all((d / f).is_file() for f in JEUX)), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique des jeux depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    for fichier, adresse in JEUX.items():
        if not (dossier / fichier).is_file():
            try:
                urllib.request.urlretrieve(adresse, dossier / fichier)
            except OSError as erreur:
                raise FileNotFoundError(
                    "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
DONNEES = dossier
print("jeux lus dans le dossier des données :", ", ".join(JEUX))

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("pandas", version("pandas"))

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook lit les jeux de sa première cellule.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = list(JEUX)                                  # les classeurs de la première cellule
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Fonctions du module du livre (`meca.py`)

Le code ci-dessous reprend, tel qu'il est écrit dans le livre, les fonctions du module employées dans ce notebook ; ce code est fourni tel quel.

In [ ]:
import pandas as pd


def lire_classeur(chemin, **options):
    """Lit un classeur et garde l'original : rend (data, df), df étant une copie de travail.

    >>> import os, tempfile
    >>> dossier = tempfile.mkdtemp()
    >>> chemin = os.path.join(dossier, "mesures.xlsx")
    >>> pd.DataFrame({"x": [1, 2, 3]}).to_excel(chemin, index=False)
    >>> data, df = lire_classeur(chemin)
    >>> df.loc[0, "x"] = 99
    >>> int(data.loc[0, "x"]), int(df.loc[0, "x"])
    (1, 99)
    """
    data = pd.read_excel(chemin, **options)
    df = data.copy()
    return data, df


def taux_manquants(df):
    """Taux de valeurs manquantes par colonne, en pourcentage, trié par ordre décroissant.

    >>> d = pd.DataFrame({"a": [1.0, None, None, 4.0], "b": [1.0, 2.0, 3.0, None], "c": [1, 2, 3, 4]})
    >>> taux_manquants(d).round(1).to_dict()
    {'a': 50.0, 'b': 25.0, 'c': 0.0}
    """
    return (df.isna().mean() * 100).sort_values(ascending=False)


def cellules_et_lignes(df):
    """Cellules manquantes et lignes touchées : rend (cellules, lignes), deux comptes qui ne se confondent pas.

    >>> d = pd.DataFrame({"a": [1.0, None, None, 4.0], "b": [1.0, 2.0, 3.0, None]})
    >>> cellules_et_lignes(d)
    (3, 3)
    """
    return int(df.isna().sum().sum()), int(df.isna().any(axis=1).sum())


def verifier_bornes(df, bornes, arreter=True):
    """Compte, par colonne, les mesures hors de leur plage physique ; `arreter` lève une erreur au premier écart.

    `bornes` associe un nom de colonne à (minimum, maximum). Les valeurs manquantes ne sont pas des écarts.

    >>> d = pd.DataFrame({"T": [20.0, 25.0, 900.0], "P": [1.0, 2.0, 3.0]})
    >>> verifier_bornes(d, {"T": (0.0, 200.0), "P": (0.0, 10.0)}, arreter=False)
    {'T': 1, 'P': 0}
    >>> verifier_bornes(d, {"T": (0.0, 200.0)})
    Traceback (most recent call last):
        ...
    ValueError: T : 1 mesure(s) hors de [0.0, 200.0]
    """
    ecarts = {}
    for colonne, (mini, maxi) in bornes.items():
        hors = int((~df[colonne].between(mini, maxi) & df[colonne].notna()).sum())
        ecarts[colonne] = hors
        if hors and arreter:
            raise ValueError(f"{colonne} : {hors} mesure(s) hors de [{mini}, {maxi}]")
    return ecarts

### Préparation

In [ ]:
import tempfile
from pathlib import Path
import pandas as pd
CHEMIN_RUL = DONNEES / "MecaIA-RUL.xlsx"
CHEMIN_SUIVI = DONNEES / "MecaIA-Suivi_Production_Machines.xlsx"
CHEMIN_PANNES = DONNEES / "MecaIA-AI4I_maintenance.xlsx"
CHEMIN_MODES = DONNEES / "MecaIA-AI4I_maintenance_TypePanne.xlsx"
pd.set_option("display.width", 100)

### Section 7.1 : un fichier de mesures se lit en une ligne.

In [ ]:
# notion: lire
data = pd.read_excel(CHEMIN_RUL)
print("colonnes :", data.columns.tolist())
print(data.head(3))
print(data.dtypes)
vue = pd.DataFrame({"type": data.dtypes, "vides": data.isna().sum()})      # type et cases vides de chaque colonne
print(vue)
assert vue.loc["niveau_sollicitation_thermique_cat", "vides"] == 3 and vue["vides"].sum() == 3
resume = data.describe()
assert resume.loc["count", "RUL_jours"] == 240 and data["RUL_jours"].dtype == "int64"

In [ ]:
# notion: dataframe
petit = pd.DataFrame({"effort_N": [1000.0, 2000.0], "essai": ["A", "B"]}, index=["e1", "e2"])
serie = petit["effort_N"]                              # une colonne est une Series
assert (
    isinstance(serie, pd.Series) and petit.index.tolist() == ["e1", "e2"] and petit.shape == (2, 2)
)
assert petit.dtypes["effort_N"] == "float64", "un type par colonne"
assert pd.Series([3, 1, 2]).index.tolist() == [0, 1, 2]

In [ ]:
# notion: csv-fr
with tempfile.TemporaryDirectory() as dossier:
    chemin_csv = Path(dossier) / "mesures.csv"
    chemin_csv.write_bytes("effort;déformation\n12,5;0,004\n13,0;0,0042\n".encode("cp1252"))
    lu = pd.read_csv(chemin_csv, sep=";", decimal=",", encoding="cp1252")
    assert lu.columns.tolist() == ["effort", "déformation"] and lu.loc[0, "effort"] == 12.5
    assert lu["déformation"].dtype == "float64", "la virgule décimale est comprise"
    # notion: ecrire
    chemin_xlsx = Path(dossier) / "sortie.xlsx"
    lu.to_excel(chemin_xlsx, index=False)
    assert pd.read_excel(chemin_xlsx).equals(lu), "écrire puis relire rend le même tableau"

## Exercice 7.1 : Lire un fichier de mesures (10 min)

Dans votre carnet, lisez `MecaIA-RUL.xlsx`. Donnez ses dimensions, la liste de ses colonnes avec leur type, puis, avec `describe`, la durée de vie moyenne des organes.

**Résultat attendu.** 240 lignes et 6 colonnes, dont une colonne de texte ; durée de vie moyenne de 199,8 jours.

**Seconde voie.** `len(data)` et `len(data.columns)` redonnent les deux dimensions, et `data["RUL_jours"].mean()` la moyenne de `describe`.

**Indication.** `pd.read_excel("donnees/MecaIA-RUL.xlsx")` rend le tableau. Son attribut `shape` donne les lignes et les colonnes, `dtypes` les types, et `describe()` la moyenne (ligne `mean`) de la colonne `RUL_jours`.

Les contrôles de la cellule lisent les variables `colonnes`, `lignes` : gardez ces noms.

In [ ]:
# Votre code :
assert (lignes, colonnes) == (240, 6), "dimensions attendues : 240 x 6"
assert (
    (lignes, colonnes) == (240, 6)
), "la fiche du jeu dit la même taille"
# Votre code :
print("Exercice 7.1 : contrôles passés")

### Section 7.2 : chaque colonne a un type et chaque ligne une adresse.

In [ ]:
# notion: copie-sauvegarde
data, df = lire_classeur(CHEMIN_RUL)               # data garde l'original, df sert au travail
df.loc[0, "RUL_jours"] = 0
assert data.loc[0, "RUL_jours"] != 0, "l'original n'est pas modifié"
df = data.copy()
# notion: colonnes
une_colonne = df["masse_vibrante_kg"]
deux_colonnes = df[["masse_vibrante_kg", "RUL_jours"]]
sans_temps = df.drop(columns=["temps_reponse_ms"])
renomme = df.rename(columns={"RUL_jours": "duree_vie_jours"})
avec_ratio = df.assign(couple_par_kw=df["couple_charge_Nm"] / df["puissance_mecanique_kw"])
numeriques = df.select_dtypes(include="number")
assert (
    isinstance(une_colonne, pd.Series) and deux_colonnes.shape == (240, 2) and sans_temps.shape == (240, 5)
)
assert "duree_vie_jours" in renomme.columns and "couple_par_kw" in avec_ratio.columns

In [ ]:
assert numeriques.shape == (240, 5), "cinq colonnes numériques, une catégorie"
# notion: index
organes = df.assign(organe=["O%03d" % k for k in range(len(df))]).set_index("organe")
assert (
    organes.index.name == "organe" and organes.loc["O005", "RUL_jours"] == df.loc[5, "RUL_jours"]
)
retour = organes.reset_index()
assert retour.columns[0] == "organe" and len(retour) == 240
assert (
    organes.reindex(["O001", "O999"]).shape == (2, 6)
), "une adresse absente donne une ligne vide"

In [ ]:
# notion: lignes
assert (
    df.iloc[0, 5] == df.loc[0, "RUL_jours"]
), "iloc par position, loc par adresse et nom de colonne"
critiques = df[df["RUL_jours"] < 100]                         # un masque choisit des lignes
chauds = df.loc[df["niveau_sollicitation_thermique_cat"].isin(["Moyen", "Élevé"]), "masse_vibrante_kg"]
tries = df.sort_values("RUL_jours")
plus_lourds = df.nlargest(3, "masse_vibrante_kg")
assert (
    len(critiques) == int((df["RUL_jours"] < 100).sum()) and tries.iloc[0]["RUL_jours"] == df["RUL_jours"].min()
)
assert df["masse_vibrante_kg"].idxmax() == plus_lourds.index[0] and len(chauds) == 118

In [ ]:
# notion: types
niveau = df["niveau_sollicitation_thermique_cat"].astype("category")
entiers_nuls = df["RUL_jours"].astype("Int64")
bruit = pd.Series(["12.5", "n/a", "13.0"])
converti = pd.to_numeric(bruit, errors="coerce")
assert sorted(niveau.cat.categories) == ["Faible", "Moyen", "Élevé"], "trois modalités"
assert (
    str(entiers_nuls.dtype) == "Int64" and converti.count() == 2 and converti.iloc[2] == 13.0
)

## Exercice 7.2 : Typer une colonne et choisir des lignes (15 min)

Sur une copie du tableau des organes, convertissez la colonne de niveau thermique en catégorie et relevez ses modalités. Gardez les organes dont le niveau est l'une de ces modalités : combien de lignes restent, sur combien ? Affichez les types des colonnes après conversion.

**Résultat attendu.** 3 modalités ; 237 lignes sur 240, soit 3 organes sans niveau ; la colonne de niveau est de type `category`.

**Seconde voie.** Sans la conversion, `isin(["Faible", "Moyen", "Élevé"])` garde le même nombre de lignes.

**Indication.** On fixe le type par `astype("category")`, et `.cat.categories` liste les modalités. Le masque `isin(modalites)` est vrai pour les niveaux connus ; on garde ces lignes avec ce masque et `len` les compte. Les cases vides ne sont dans aucune modalité.

Les contrôles de la cellule lisent les variables `connues`, `modalites` : gardez ces noms.

In [ ]:
# Votre code :
assert len(modalites) == 3 and len(connues) == 237, "lignes complètes attendues : 237"
print("Exercice 7.2 : contrôles passés")

### Section 7.3 : les valeurs manquantes se comptent avant de se retirer.

In [ ]:
# notion: manquants
suivi = pd.read_excel(CHEMIN_SUIVI)
masque_vide = suivi.isna()                                # un tableau de True et False, de la forme du jeu
complet = suivi.dropna()                                  # retire toute ligne qui porte au moins un vide
print("forme du suivi :", suivi.shape, " après dropna :", complet.shape)
# notion: taux-manquants
vides_par_colonne = suivi.isna().sum()
taux = (suivi.isna().mean() * 100).sort_values(ascending=False)
cellules_vides = int(vides_par_colonne.sum())
lignes_touchees = int(suivi.isna().any(axis=1).sum())
print(taux.head(4).round(2).to_string())
print(f"cellules vides : {cellules_vides} ; lignes touchées : {lignes_touchees} ; lignes complètes : {len(complet)}")
assert masque_vide.shape == suivi.shape and len(complet) == len(suivi) - lignes_touchees
assert (
    cellules_et_lignes(suivi) == (cellules_vides, lignes_touchees)
), "le module compte comme le script"

In [ ]:
assert taux_manquants(suivi).equals(taux), "le module classe comme le script"

## Exercice 7.3 : Cellules manquantes du suivi de production (15 min)

Lisez le suivi de production. Donnez ses dimensions, puis comptez les cellules vides de chaque colonne et au total. Comptez les lignes qui portent au moins un vide, avec `isna().any(axis=1)`, puis les lignes que `dropna` conserve. Quelles colonnes portent les vides ?

**Résultat attendu.** 20 000 lignes et 14 colonnes ; 6 043 cellules vides, toutes dans 3 colonnes de capteurs (température en °C, vibrations en mm/s, consommation en kWh/h) ; 5 462 lignes touchées, 14 538 lignes complètes.

**Seconde voie.** Lignes touchées plus lignes complètes redonnent 20 000 ; la somme des vides par colonne égale celle du tableau de booléens entier.

**Indication.** `suivi.isna().sum()` compte les vides par colonne, et sa somme le total. `suivi.isna().any(axis=1).sum()` compte les lignes touchées, `len(suivi.dropna())` les lignes complètes. Seules trois colonnes de capteurs ont des vides : les cellules vides sont moins nombreuses que les lignes touchées multipliées par trois.

Les contrôles de la cellule lisent la variable `colonnes_vides` : gardez ce nom.

In [ ]:
# Votre code :
assert suivi.shape == (20000, 14), "dimensions attendues : 20000 x 14"
assert (
    suivi.shape == (20000, 14)
), "la fiche du jeu dit la même taille"
assert cellules_vides == 6043, "valeurs manquantes attendues : 6043"
assert colonnes_vides.shape[0] == 3, "trois colonnes de capteurs portent les vides"
print("Exercice 7.3 : contrôles passés")

### Section 7.4 : une mesure doit rester cohérente.

In [ ]:
# notion: coherence
pannes = pd.read_excel(CHEMIN_PANNES)
modes = pd.read_excel(CHEMIN_MODES)
print("pannes (0 ou 1) :", pannes["Panne"].unique().tolist(), " effectifs :", pannes["Panne"].value_counts().tolist())
assert pannes["Panne"].nunique() == 2 and pannes.duplicated().sum() == 0
assert pannes["Panne"].value_counts()[1] == 339

In [ ]:
# notion: controle-borne
bornes = {"T_palier_C": (50.0, 130.0), "Vitesse_rot_tr_min": (500.0, 4000.0), "Couple_Nm": (0.0, 100.0),
          "Vibration_mm_s": (0.0, 25.0)}
ecarts = verifier_bornes(pannes, bornes, arreter=False)
hors_plage = pannes["T_palier_C"].between(50.0, 130.0) | pannes["T_palier_C"].isna()
assert (
    all(v == 0 for v in ecarts.values()) and hors_plage.all()
), "aucune mesure hors de sa plage physique"

try:
    verifier_bornes(pannes, {"Vibration_mm_s": (0.0, 10.0)})
    arret = "aucune erreur"
except ValueError as erreur:
    arret = str(erreur)

print("contrôle qui arrête le calcul :", arret)
assert arret.startswith("Vibration_mm_s :")

## Exercice 7.4 : Trois cent trente-neuf pannes, trois cent soixante-treize modes (défi, 25 min)

Le fichier des pannes compte 339 pannes (la colonne `Panne` vaut $1$). Le fichier des modes porte une colonne d'indicateurs par mode : la somme de ses cinq colonnes donne 373 modes. Les deux fichiers portent sur les mêmes relevés, mais le total des modes s'écarte de 34 du nombre de pannes. Expliquez-le en comptant trois nombres : les pannes qui n'ont aucun mode actif, les modes actifs sur une ligne sans panne, et les modes comptés en plus sur les lignes qui en portent plusieurs. Vérifiez que le bilan retombe sur l'écart.

**Résultat attendu.** 9 pannes sans mode, 18 modes sans panne, 25 modes en surplus : $339- 9+ 18 + 25= 373$.

**Seconde voie.** La somme des cinq colonnes, d'un seul bloc, redonne le total des modes, et la colonne `Panne` du fichier des modes compte les mêmes 339 pannes que celle des relevés.

**Indication.** Les colonnes de modes sont celles dont le nom commence par `Panne_`. Leur somme ligne à ligne (`sum(axis=1)`) donne le nombre de modes de chaque ligne. Un masque `(Panne == 1) & (nb_modes == 0)` compte les pannes sans mode, `(Panne == 0) & (nb_modes >= 1)` les modes sans panne ; le surplus est la somme de `nb_modes - 1` sur les lignes à deux modes ou plus. Le bilan est `339 - 9 + 18 + 25 = 373` (figure du chapitre).

Les contrôles de la cellule lisent les variables `mode_sans_panne`, `plusieurs_modes`, `sans_mode`, `surplus`, `total_modes`, `total_pannes` : gardez ces noms.

In [ ]:
# Votre code :
assert plusieurs_modes == 24, "24 lignes portent deux ou trois modes à la fois"
assert total_pannes == 339 and total_modes == 373, "339 pannes contre 373 modes de panne"
assert total_pannes == int(modes["Panne"].sum()), "les deux fichiers comptent les mêmes pannes"
assert (
    (pannes["time"] == modes["time"]).all()
), "les deux fichiers sont alignés ligne à ligne sur le temps"
assert (
    total_pannes - sans_mode + mode_sans_panne + surplus == total_modes
), "le bilan des écarts explique les 34"
assert total_modes - total_pannes == 34
print("Exercice 7.4 : contrôles passés")

## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F07-1 : Lecture des mesures (`pandas.read_csv`)

In [ ]:
import pandas as pd
import io

# Fichier texte avec deux lignes d'en-tête et trois mesures
texte = "Capteur\nDate\nt_ms\tv_mms\n0.0\t0.0\n2.5\t-12.4\n5.0\t8.3\n"
tableau = pd.read_csv(io.StringIO(texte), sep="\t", skiprows=2)
vitesse_max = tableau["v_mms"].abs().max()    # amplitude extrême
print(f"Vitesse maximale : {vitesse_max:.1f} mm/s".replace(".", ","))
print(tableau.shape == (3, 2))                # contrôle des dimensions